# Unidad 4 · Kimball (Notebook 2)

## Construccion de fact_ventas (SQL Server)

Este notebook implementa la carga de la tabla de hechos `dw.fact_ventas`, apoyandose en las dimensiones creadas en el notebook anterior (`dim_tiempo`, `dim_cliente`, `dim_producto`, `dim_vendedor`, `dim_ciudad`).

Origen: `WideWorldImporters`
Destino: `WideWorldImportersDW2`

**Proceso de negocio:** ventas facturadas.

**Granularidad:** una fila por linea de factura (`Sales.InvoiceLines`).

**Medidas:**

- Aditivas: `cantidad`, `monto_neto`, `margen_bruto`.
- No aditivas: `precio_unitario` (no se suma, se promedia).

Criterios Kimball aplicados:

1. Tabla de hechos al grano mas fino (linea de factura).
2. Resolucion de claves sustitutas (surrogate keys) de todas las dimensiones.
3. Resolucion temporal de SCD Tipo 2 para `dim_cliente` y `dim_producto` (se busca la version vigente a la fecha de la factura).
4. Dimension degenerada: `numero_factura` + `numero_linea`.

> Nota: para fines pedagogicos, el flujo esta simplificado respecto al repositorio oficial de Microsoft.


## 1) Dependencias

Si hace falta instalar paquetes, descomentar y ejecutar la siguiente linea:

```python
# %pip install pandas sqlalchemy pyodbc
```


In [ ]:
import urllib
import urllib.parse

import pandas as pd
import pyodbc
from sqlalchemy import create_engine, text

print('Librerias importadas correctamente')

## 2) Parametros de conexion

En este ejemplo usamos autenticacion SQL.


In [ ]:
# Conexion SQL Server
SERVIDOR  = r'localhost\SQLEXPRESS02'
BD_ORIGEN = 'WideWorldImporters'
BD_DESTINO = 'WideWorldImportersDW2'
USUARIO = 'ingesta_reader'
CLAVE = '123456789'

# Seleccion automatica de driver
drivers = pyodbc.drivers()
if 'ODBC Driver 18 for SQL Server' in drivers:
    DRIVER = 'ODBC Driver 18 for SQL Server'
elif 'ODBC Driver 17 for SQL Server' in drivers:
    DRIVER = 'ODBC Driver 17 for SQL Server'
else:
    raise RuntimeError('No se encontro ODBC Driver 17/18 para SQL Server')

print(f'Driver: {DRIVER}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')

In [ ]:
def construir_engine(base_datos: str):
    conn_str = (
        f'DRIVER={{{DRIVER}}};'
        f'SERVER={SERVIDOR};'
        f'DATABASE={base_datos};'
        f'UID={USUARIO};'
        f'PWD={CLAVE};'
        'TrustServerCertificate=yes;'
    )
    params = urllib.parse.quote_plus(conn_str)
    return create_engine(f'mssql+pyodbc:///?odbc_connect={params}')

engine_origen = construir_engine(BD_ORIGEN)
engine_destino = construir_engine(BD_DESTINO)

with engine_origen.connect() as c1:
    v1 = c1.execute(text('SELECT DB_NAME() AS bd')).mappings().first()
with engine_destino.connect() as c2:
    v2 = c2.execute(text('SELECT DB_NAME() AS bd')).mappings().first()

print(f'Conexion OK origen : {v1["bd"]}')
print(f'Conexion OK destino: {v2["bd"]}')

## 3) DDL de dw.fact_ventas

Tabla de hechos transaccional con grano de linea de factura. Incluye FK logicas hacia las 5 dimensiones y un indice unico sobre la dimension degenerada (`numero_factura` + `numero_linea`) para evitar cargas duplicadas.


In [ ]:
sql_ddl_fact = '''
IF OBJECT_ID('dw.fact_ventas', 'U') IS NULL
BEGIN
    CREATE TABLE dw.fact_ventas (
        id_fact_venta_sk    BIGINT IDENTITY(1,1) PRIMARY KEY,
        id_tiempo           INT            NOT NULL,
        id_cliente_sk       INT            NOT NULL,
        id_producto_sk      INT            NOT NULL,
        id_vendedor_sk      INT            NOT NULL,
        id_ciudad_sk        INT            NOT NULL,
        numero_factura      INT            NOT NULL,
        numero_linea        INT            NOT NULL,
        cantidad            DECIMAL(18,4)  NOT NULL,
        precio_unitario     DECIMAL(18,4)  NOT NULL,
        monto_neto          DECIMAL(18,2)  NOT NULL,
        costo_estimado      DECIMAL(18,2)  NULL,
        margen_bruto        DECIMAL(18,2)  NULL,
        fecha_carga         DATETIME2      NOT NULL DEFAULT SYSDATETIME(),

        CONSTRAINT fk_fv_tiempo    FOREIGN KEY (id_tiempo)      REFERENCES dw.dim_tiempo (id_tiempo),
        CONSTRAINT fk_fv_cliente   FOREIGN KEY (id_cliente_sk)  REFERENCES dw.dim_cliente (id_cliente_sk),
        CONSTRAINT fk_fv_producto  FOREIGN KEY (id_producto_sk) REFERENCES dw.dim_producto (id_producto_sk),
        CONSTRAINT fk_fv_vendedor  FOREIGN KEY (id_vendedor_sk) REFERENCES dw.dim_vendedor (id_vendedor_sk),
        CONSTRAINT fk_fv_ciudad    FOREIGN KEY (id_ciudad_sk)   REFERENCES dw.dim_ciudad (id_ciudad_sk)
    );

    CREATE UNIQUE INDEX ux_fact_ventas_factura_linea ON dw.fact_ventas (numero_factura, numero_linea);
    CREATE INDEX ix_fact_ventas_tiempo   ON dw.fact_ventas (id_tiempo);
    CREATE INDEX ix_fact_ventas_producto ON dw.fact_ventas (id_producto_sk);
    CREATE INDEX ix_fact_ventas_cliente  ON dw.fact_ventas (id_cliente_sk);
END;
'''

with engine_destino.begin() as conn:
    conn.execute(text(sql_ddl_fact))

print('Tabla dw.fact_ventas verificada/creada correctamente')

## 4) Extraccion de lineas de factura desde WideWorldImporters

Fuente: `Sales.Invoices` + `Sales.InvoiceLines` + `Sales.Customers` (para resolver la ciudad de entrega del cliente).

Claves naturales extraidas: `CustomerID`, `StockItemID`, `SalespersonPersonID`, `DeliveryCityID`. Estas se resuelven luego contra las dimensiones del destino para obtener las claves sustitutas.


In [ ]:
sql_ventas_origen = '''
SELECT
    i.InvoiceID              AS numero_factura,
    il.InvoiceLineID          AS numero_linea,
    i.InvoiceDate              AS fecha_factura,
    i.CustomerID                AS id_cliente_nk,
    il.StockItemID                AS id_producto_nk,
    i.SalespersonPersonID          AS id_vendedor_nk,
    c.DeliveryCityID                 AS id_ciudad_nk,
    il.Quantity                        AS cantidad,
    il.UnitPrice                        AS precio_unitario,
    il.LineProfit                        AS margen_bruto
FROM Sales.Invoices i
JOIN Sales.InvoiceLines il
    ON i.InvoiceID = il.InvoiceID
JOIN Sales.Customers c
    ON i.CustomerID = c.CustomerID;
'''

with engine_origen.connect() as conn:
    df_ventas_src = pd.read_sql(sql_ventas_origen, conn)

# Medidas aditivas derivadas: monto neto (antes de impuestos) y costo estimado
df_ventas_src['monto_neto'] = df_ventas_src['cantidad'] * df_ventas_src['precio_unitario']
df_ventas_src['costo_estimado'] = df_ventas_src['monto_neto'] - df_ventas_src['margen_bruto']
df_ventas_src['fecha_factura'] = pd.to_datetime(df_ventas_src['fecha_factura']).dt.normalize()
df_ventas_src['id_tiempo'] = df_ventas_src['fecha_factura'].dt.strftime('%Y%m%d').astype(int)

print(f'Lineas de factura extraidas desde origen: {len(df_ventas_src)}')
df_ventas_src.head()

## 5) Resolucion de claves sustitutas (surrogate keys)

- `dim_vendedor` y `dim_ciudad`: son dimensiones tipo 1, se resuelven con un merge directo por clave natural.
- `dim_cliente` y `dim_producto`: son SCD Tipo 2, se resuelve la version vigente a la fecha de la factura filtrando por `fecha_inicio_vigencia` / `fecha_fin_vigencia`.


In [ ]:
with engine_destino.connect() as conn:
    df_dim_vendedor = pd.read_sql('SELECT id_vendedor_sk, id_vendedor_nk FROM dw.dim_vendedor', conn)
    df_dim_ciudad = pd.read_sql('SELECT id_ciudad_sk, id_ciudad_nk FROM dw.dim_ciudad', conn)
    df_dim_cliente = pd.read_sql(
        'SELECT id_cliente_sk, id_cliente_nk, fecha_inicio_vigencia, fecha_fin_vigencia FROM dw.dim_cliente', conn
    )
    df_dim_producto = pd.read_sql(
        'SELECT id_producto_sk, id_producto_nk, fecha_inicio_vigencia, fecha_fin_vigencia FROM dw.dim_producto', conn
    )

for col in ('fecha_inicio_vigencia', 'fecha_fin_vigencia'):
    df_dim_cliente[col] = pd.to_datetime(df_dim_cliente[col])
    df_dim_producto[col] = pd.to_datetime(df_dim_producto[col])

df_fact = df_ventas_src.copy()

# Dimensiones tipo 1: merge directo por clave natural
df_fact = df_fact.merge(df_dim_vendedor, on='id_vendedor_nk', how='left')
df_fact = df_fact.merge(df_dim_ciudad, on='id_ciudad_nk', how='left')

# Dimension SCD2 cliente: version vigente a la fecha de la factura
df_fact = df_fact.merge(df_dim_cliente, on='id_cliente_nk', how='left', suffixes=('', '_cliente'))
vigente_cliente = (
    (df_fact['fecha_factura'] >= df_fact['fecha_inicio_vigencia']) &
    (df_fact['fecha_factura'] <= df_fact['fecha_fin_vigencia'])
)
df_fact = df_fact[vigente_cliente].drop(columns=['fecha_inicio_vigencia', 'fecha_fin_vigencia'])

# Dimension SCD2 producto: version vigente a la fecha de la factura
df_fact = df_fact.merge(df_dim_producto, on='id_producto_nk', how='left', suffixes=('', '_producto'))
vigente_producto = (
    (df_fact['fecha_factura'] >= df_fact['fecha_inicio_vigencia']) &
    (df_fact['fecha_factura'] <= df_fact['fecha_fin_vigencia'])
)
df_fact = df_fact[vigente_producto].drop(columns=['fecha_inicio_vigencia', 'fecha_fin_vigencia'])

# Filas sin resolucion completa de claves sustitutas se descartan y se reportan
claves_sk = ['id_tiempo', 'id_cliente_sk', 'id_producto_sk', 'id_vendedor_sk', 'id_ciudad_sk']
mascara_incompletas = df_fact[claves_sk].isna().any(axis=1)
df_fact_incompletas = df_fact[mascara_incompletas].copy()
df_fact = df_fact[~mascara_incompletas].copy()

for col in ['id_cliente_sk', 'id_producto_sk', 'id_vendedor_sk', 'id_ciudad_sk']:
    df_fact[col] = df_fact[col].astype(int)

print(f'Lineas con claves sustitutas resueltas : {len(df_fact)}')
print(f'Lineas descartadas (claves incompletas) : {len(df_fact_incompletas)}')

## 6) Carga incremental de fact_ventas

Se insertan unicamente las lineas de factura (`numero_factura` + `numero_linea`) que todavia no existen en `dw.fact_ventas`, evitando duplicados en re-ejecuciones.


In [ ]:
with engine_destino.connect() as conn:
    df_fact_existente = pd.read_sql(
        'SELECT numero_factura, numero_linea FROM dw.fact_ventas', conn
    )

if df_fact_existente.empty:
    df_fact_nuevas = df_fact.copy()
else:
    df_fact_nuevas = df_fact.merge(
        df_fact_existente,
        on=['numero_factura', 'numero_linea'],
        how='left',
        indicator=True
    )
    df_fact_nuevas = df_fact_nuevas[df_fact_nuevas['_merge'] == 'left_only'].drop(columns=['_merge'])

columnas_fact = [
    'id_tiempo', 'id_cliente_sk', 'id_producto_sk', 'id_vendedor_sk', 'id_ciudad_sk',
    'numero_factura', 'numero_linea', 'cantidad', 'precio_unitario',
    'monto_neto', 'costo_estimado', 'margen_bruto'
]

if not df_fact_nuevas.empty:
    df_fact_nuevas[columnas_fact].to_sql(
        'fact_ventas', engine_destino, schema='dw', if_exists='append', index=False
    )

print(f'Lineas nuevas insertadas en fact_ventas: {len(df_fact_nuevas)}')

## 7) Validaciones y consultas de control

- Volumen total y por mes.
- Margen por producto.
- Reconciliacion de totales contra el origen (`WideWorldImporters`).


In [ ]:
sql_volumen_mensual = '''
SELECT t.anio_numero AS anio, t.mes_numero AS mes, COUNT(*) AS filas_fact
FROM dw.fact_ventas f
JOIN dw.dim_tiempo t ON f.id_tiempo = t.id_tiempo
GROUP BY t.anio_numero, t.mes_numero
ORDER BY t.anio_numero, t.mes_numero;
'''

sql_margen_producto = '''
SELECT TOP 20
    p.nombre_producto,
    SUM(f.monto_neto) AS venta,
    SUM(f.margen_bruto) AS margen
FROM dw.fact_ventas f
JOIN dw.dim_producto p ON f.id_producto_sk = p.id_producto_sk
GROUP BY p.nombre_producto
ORDER BY venta DESC;
'''

with engine_destino.connect() as conn:
    df_volumen_mensual = pd.read_sql(sql_volumen_mensual, conn)
    df_margen_producto = pd.read_sql(sql_margen_producto, conn)

print('Volumen mensual de fact_ventas')
display(df_volumen_mensual)

print('Margen por producto (Top 20)')
display(df_margen_producto)

In [ ]:
# Reconciliacion: total de lineas y monto neto en destino vs. total extraido desde el origen
with engine_destino.connect() as conn:
    resumen_destino = conn.execute(text(
        'SELECT COUNT(*) AS filas, SUM(monto_neto) AS monto_neto FROM dw.fact_ventas'
    )).mappings().first()

resumen_origen = {
    'filas': len(df_ventas_src),
    'monto_neto': df_ventas_src['monto_neto'].sum()
}

print(f'Origen  -> filas: {resumen_origen["filas"]}, monto_neto: {resumen_origen["monto_neto"]:.2f}')
print(f'Destino -> filas: {resumen_destino["filas"]}, monto_neto: {resumen_destino["monto_neto"]:.2f}')

if len(df_fact_incompletas) > 0:
    print(f'Atencion: {len(df_fact_incompletas)} lineas no se cargaron por claves sustitutas incompletas')